In [69]:
from qiskit import QuantumRegister, QuantumCircuit, ClassicalRegister
from qiskit.quantum_info import Statevector
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as Sampler, EstimatorV2 as Estimator
# from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

import numpy as np

In [28]:
block_size = 3
def draw(strings):
    height = len(strings)
    width = len(strings[0])
    for j in range(height):
        cs = strings[j]
        row = ""
        for i in range(width):
            c = cs[i]
            c = " " if c == "0" else c
            row = row + c*block_size
        for n in range(block_size):
            print(row)

In [156]:
import math
class Point:
    def __init__(self, x, y, color):
        self.x = x
        self.y = y
        self.color = color

    def to_string(self):
        return str(self.color)

    def to_linear(self, width, height):
        return self.y * width + self.x

In [157]:
class Image:
    def __init__(self, width, height, points=[], max_color=8):
        self.width = width 
        self.height = height
        self.max_color = max_color
        self.points = points

    def normalized_at(self, x, y):
        pt = self.points[(x,y)]
        if pt:
            return (pt.to_linear(width=self.width, height=self.height), pt.color / float(self.max_color))

    def normalized(self):
        return [self.normalized_at(x,y) for (x,y) in self.points.keys()]

    def draw(self):
        rows = []
        for j in range(self.height):
            row = ''.join([self.points[(i,j)].to_string() for i in range(self.width)])
            rows.append(row)
        draw(rows)

    @staticmethod
    def from_string(string, height=1, y=0):
        points = {}
        for i, s in enumerate(string):
            points[(i, y)] = Point(i, y, int(s))
        return points

    @staticmethod
    def from_strings(strings):
        points = {}
        for j, ss in enumerate(strings):
            points.update(Image.from_string(ss, len(strings), j))
        return Image(width=len(strings[0]), height=len(strings), points=points)

In [160]:
Image.from_strings(["011", "100", "321"]).draw()

   111111
   111111
   111111
111      
111      
111      
333222111
333222111
333222111


In [164]:
def bin_pts(n):
    pts = []
    i = 0
    while n > 0:
        if n % 2 == 1:
            pts.append(i)
        n = n // 2
        i = i + 1
    return pts

In [271]:
import numpy as np
def frqi_draw_point(qc, qr, rotation, pt_n, pt_c):
    toggles = bin_pts(pt_n)
    for n in toggles:
        qc.x(qr[n])
    qc.mcry(2*np.arcsin(pt_c), qr, rotation)
    for n in toggles:
        qc.x(qr[n])

In [226]:
image = Image.from_strings(["080", "400", "080"])

In [272]:
qr = QuantumRegister(4)
rotation = QuantumRegister(1)
qc = QuantumCircuit(qr, rotation)

qc.h(qr)
qc.barrier()

for (pt_n, pt_c) in image.normalized():
    frqi_draw_point(qc, qr, rotation, pt_n, pt_c)

qc.draw(output='mpl')

qc.measure_all()

In [273]:
sampler = Sampler()
pm = generate_preset_pass_manager(3, AerSimulator())
isa_c = pm.run(qc)
result = sampler.run([isa_c]).result()
data = result[0].data
data

DataBin(meas=BitArray(<shape=(), num_shots=1024, num_bits=5>))

In [274]:
data['meas'].get_counts()

{'00001': 63,
 '11000': 65,
 '00000': 80,
 '11110': 63,
 '01101': 64,
 '01100': 66,
 '01111': 64,
 '01011': 71,
 '00110': 59,
 '00100': 59,
 '00010': 54,
 '00111': 64,
 '01001': 59,
 '00011': 53,
 '01010': 62,
 '00101': 61,
 '11100': 17}

In [275]:
def parse_bitstring(n):
    c = int(n[0])
    m = 0
    for s in n[-1:0:-1]:
        m = m*2 + int(s)
    return (m, c)

In [276]:
def parse_counts(data, image):
    cts = data.get_counts()
    parsed = {}
    for ct in cts:
        (n, c) = parse_bitstring(ct)
        cs = parsed.get(n, [])
        cs.append((c, cts[ct]))
        parsed[n] = cs
    for p in parsed:
        vls = parsed[p]
        n = 0
        total = 0
        for (c, v) in vls:
            n = n + v
            total = total + c*v
        parsed[p] = np.sqrt(total / n) * image.max_color
    return parsed

In [277]:
def to_strings(data, image):
    parsed = parse_counts(data, image)
    row = ""
    rows = []
    for i in range(len(parsed)):
        c = parsed.get(i)
        if c:
            row = row + str(round(c))
        else:
            row = row + "0"
        if (i + 1) % image.width == 0:
            rows.append(row)
            row = ""
    return Image.from_strings(rows)

In [278]:
i2 = to_strings(data['meas'], image)
i2.draw()

   888   
   888   
   888   
444      
444      
444      
   888   
   888   
   888   
         
         
         
         
         
         
